In [ ]:
# @title Множество Жюлия 
import numpy as np
import matplotlib.pyplot as plt
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

# Параметры изображения
# @markdown Параметры изображения:
width = 30720 # 3840 # @param {type: "integer"} 
height =17280 # 2160 # @param {type: "integer"} 
# @markdown Кол-во итераций для остановки проверки:
max_iter = 256 # @param {type: "integer"}

# Границы на комплексной плоскости
# Коэффициент масштабирования (scale: 1.0 - исходный масштаб, < 1.0 - приближение, > 1.0 - отдаление)
# @markdown Коэффициент масштабирования:
scale = 0.2 # @param {type:"number",title="12"} #2  #Adjust this value to change the zoom level
# Исходные координаты (можете менять, чтобы смотреть разные части множества)
# @markdown Исходные координаты:
center_x = 0.0 #  @param {type:"number"}
center_y =  0.0 # @param {type: "number"}
zoomed_width, zoomed_height = 0.25, 0.175  # Initial width and height for the zoomed area

# Вычисляем новые координаты на основе scale
xmin, xmax = center_x - zoomed_width / scale, center_x + zoomed_width / scale
ymin, ymax = center_y - zoomed_height / scale, center_y + zoomed_height / scale
# Преобразуем в массивы numpy для передачи на GPU
image = np.zeros((height, width), dtype=np.float32)

# Задаем параметр c для множества Жюлиа
c_real, c_imag = -0.8 , 0.156 # Пример: c = -0.7 + 0.27015j ; -0.8 + 0.156i;

# CUDA-код для вычисления множества Жюлиа
julia_kernel = SourceModule("""
__global__ void julia(float *image, int width, int height, 
                      float xmin, float xmax, float ymin, float ymax, 
                      float c_real, float c_imag, int max_iter) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width || y >= height) return;

    float real = xmin + x * (xmax - xmin) / width;
    float imag = ymin + y * (ymax - ymin) / height;

    float z_real = real;
    float z_imag = imag;

    int value = 0;
    for (int i = 0; i < max_iter; i++) {
        float z_real2 = z_real * z_real;
        float z_imag2 = z_imag * z_imag;
        value = i;
        // Условие выхода за границу
        if (z_real2 + z_imag2 > 4.0f) {
            value = i;
            break;
        }

        // Итерация функции для множества Жюлиа
        float new_real = z_real2 - z_imag2 + c_real;
        float new_imag = 2.0f * z_real * z_imag + c_imag;
        
        z_real = new_real;
        z_imag = new_imag;
    }

    image[y * width + x] = (float)value;
}
""")

# Получаем функцию из модуля
julia = julia_kernel.get_function("julia")

# Выделяем память на GPU для изображения
image_gpu = cuda.mem_alloc(image.nbytes)

# Задаем параметры сетки и блоков
block_dim = (16, 16, 1)  # Блоки 16x16
grid_dim = ((width + block_dim[0] - 1) // block_dim[0],
            (height + block_dim[1] - 1) // block_dim[1])

print("init compute to GPU")
# Запускаем вычисление на GPU
julia(image_gpu, np.int32(width), np.int32(height),
      np.float32(xmin), np.float32(xmax), np.float32(ymin), np.float32(ymax),
      np.float32(c_real), np.float32(c_imag), np.int32(max_iter),
      block=block_dim, grid=grid_dim)

print("success computed GPU")
# Копируем результат с GPU на CPU
print("init convert GPU mem to CPU mem")
cuda.memcpy_dtoh(image, image_gpu)
print("success convert GPU to CPU")
# Настройка размера фигуры в дюймах для соответствия разрешению 4K
fig = plt.figure(figsize=(307.2, 172.8))  # размеры в дюймах
# Визуализируем изображение с палитрой "prism"
plt.imshow(image, extent=(xmin, xmax, ymin, ymax), cmap="hot")  # Используем палитру "twilight"
#plt.colorbar(label="Итерации до выхода за пределы")
plt.title("Множество Жюлиа с использованием PyCUDA")
plt.xlabel("Re(z)")
plt.ylabel("Im(z)")
print("init save image to png")
# Сохранение изображения с высоким качеством
plt.savefig("julia_set_high_quality.png", dpi=150, bbox_inches="tight", format="png")
print("success save image to png")

#plt.show()


init compute to GPU
success computed GPU
init convert GPU mem to CPU mem
success convert GPU to CPU
init save image to png
success save image to png
